<a href="https://colab.research.google.com/github/amankiitg/LLM_Prod/blob/main/Reditt_Text_Clustering_and_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Text Clustering and Topic Modeling</h1>
<i>Clustering documents using a wide variety of language models.</i>



### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
!pip install asyncpraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import asyncio
import asyncpraw
import nest_asyncio
import json
import numpy as np
import re
import string

# --------------------
# Text cleaning helper
# --------------------
def clean_text(text):
    """Cleans text by removing URLs, punctuation, and lowercasing."""
    text = re.sub(r'http\S+|https\S+', '', text)   # Remove URLs
    text = text.replace('\n', ' ')                 # Remove newlines
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    return text.lower().strip()

def extract_comment_bodies(comments):
    """Recursively extracts bodies and scores for thresholding."""
    bodies, scores = [], []
    for comment in comments:
        bodies.append(comment['body'])
        scores.append(comment['score'])
        if comment.get('replies'):
            child_bodies, child_scores = extract_comment_bodies(comment['replies'])
            bodies.extend(child_bodies)
            scores.extend(child_scores)
    return bodies, scores

# --------------------
# Main Scraper
# --------------------
async def scrape():
    async with asyncpraw.Reddit(
        client_id="glnTvdUs1KFycKy5vEsOig",
        client_secret="ET1IgiyCztIxhXSwnFIYbTs3eq2nRQ",
        user_agent="Topic Modeler"
    ) as reddit:

        url = "https://www.reddit.com/r/ChatGPT/comments/1mkae1l/gpt5_ama_with_openais_sam_altman_and_some_of_the/"
        submission = await reddit.submission(url=url)

        async def extract_comment(comment):
            return {
                "author": str(comment.author),
                "body": comment.body,
                "score": comment.score,
                "replies": [await extract_comment(reply) async for reply in comment.replies]
            }

        # Build post object
        post_data = {
            "title": submission.title,
            "author": str(submission.author),
            "score": submission.score,
            "url": submission.url,
            "selftext": submission.selftext,
            "comments": []
        }

        # Load comments
        await submission.comments.replace_more(limit=None)
        for top_comment in submission.comments:
            post_data["comments"].append(await extract_comment(top_comment))

        # Flatten scores for median threshold
        _, all_scores = extract_comment_bodies(post_data["comments"])
        median_score = np.median(all_scores) if all_scores else 0

        # --------------------
        # Filter + transform comments
        # --------------------
        def filter_and_transform(comments):
            """Filter by median score and create title/abstract."""
            result = []
            for c in comments:
                if c['score'] >= median_score:
                    result.append({
                        "title": clean_text(c['body'])[:60],   # First 60 chars as title
                        "abstract": clean_text(c['body']),     # Full cleaned body
                        "score": c['score'],
                        "author": c['author']
                    })
                if c.get('replies'):
                    result.extend(filter_and_transform(c['replies']))
            return result

        post_data["comments"] = filter_and_transform(post_data["comments"])
        post_data["median_score_threshold"] = median_score

        # Dump JSON
        with open("reddit_post_filtered.json", "w", encoding="utf-8") as f:
            json.dump(post_data, f, indent=2, ensure_ascii=False)

        print("✅ Filtered post data saved to reddit_post_filtered.json")

# --------------------
# Run safely with nest_asyncio
# --------------------
nest_asyncio.apply()
asyncio.run(scrape())


✅ Filtered post data saved to reddit_post_filtered.json


/tmp/ipython-input-3660819704.py:65: DeprecationWarning: Using CommentForest as an asynchronous iterator has been deprecated and will be removed in a future version.
  post_data["comments"].append(await extract_comment(top_comment))
/tmp/ipython-input-3660819704.py:49: DeprecationWarning: Using CommentForest as an asynchronous iterator has been deprecated and will be removed in a future version.
  "replies": [await extract_comment(reply) async for reply in comment.replies]


# **Load Reditt Data**

In [1]:
import json

with open("/content/reddit_post_filtered.json", "r", encoding="utf-8") as f:
    reddit_data = json.load(f)

# Recursive extractor for title and abstract
def extract_title_abstract(comments):
    result = []
    for c in comments:
        result.append({
            "title": c["title"],
            "abstract": c["abstract"]
        })
        if c.get("replies"):
            result.extend(extract_title_abstract(c["replies"]))
    return result

# Extract only title and abstract
titles_and_abstracts = extract_title_abstract(reddit_data["comments"])

# Display first 10
for item in titles_and_abstracts[:10]:
    print(item)

# # Extract metadata and convert to standard list
abstracts = [t['abstract'] for t in titles_and_abstracts]
titles = [t['title'] for t in titles_and_abstracts]

{'title': 'i see that em dash', 'abstract': 'i see that em dash'}
{'title': 'its not an em dash — its a pause that demands attention', 'abstract': 'its not an em dash — its a pause that demands attention'}
{'title': 'can you mark the other cohosts their answers are buried and ', 'abstract': 'can you mark the other cohosts their answers are buried and unhighlighted and the only way you can find them at present is by checking each profiles comment history manually   currently the filter for answered misses every other cohost reply other than sam altman making the ama much harder to navigate  or provide some other way to see them thanks'}
{'title': 'only uopenai can i think theres a limit to the number of par', 'abstract': 'only uopenai can i think theres a limit to the number of participants'}
{'title': 'ah thats a bother perhaps sticky links to them in a comment ', 'abstract': 'ah thats a bother perhaps sticky links to them in a comment after the ama ends its kind of silly that theres b

## **BERTopic: A Modular Topic Modeling Framework**

In [2]:
!pip install -q bertopic openai datasets datamapplot

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rapids-dask-dependency 25.6.0 requires dask==2025.5.0, but you have dask 2024.12.1 which is incompatible.
rapids-dask-dependency 25.6.0 requires distribu

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration, OpenAI
from transformers import pipeline
from wordcloud import WordCloud
import openai
from copy import deepcopy
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from google.colab import userdata
import datamapplot
import re


def find_best_k(embeddings, k_min=2, k_max=15, random_state=42):
    """Return best k based on silhouette score"""
    best_score = -1
    best_k = k_min
    for k in range(k_min, k_max + 1):
        kmeans = KMeans(n_clusters=k, random_state=random_state).fit(embeddings)
        score = silhouette_score(embeddings, kmeans.labels_)
        if score > best_score:
            best_score = score
            best_k = k
    print(f"✅ Best k for KMeans: {best_k} with silhouette {best_score:.3f}")
    return best_k

# ==================== Helper Functions ====================
def topic_differences(model, original_topics, nr_topics=5):
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]
    return df

def create_wordcloud(model, topic, save_path=None):
    """Generate a wordcloud for a given topic if it exists."""
    topic_words = model.get_topic(topic)
    if not topic_words:  # Returns False if topic does not exist
        print(f"⚠️ Topic {topic} does not exist. Skipping wordcloud.")
        return

    plt.figure(figsize=(10,5))
    text = {word: value for word, value in topic_words}
    wc = WordCloud(background_color="white", max_words=1000, width=1600, height=800)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()


def save_document_visualization(model, documents, reduced_embeddings, save_path=None):
    """
    Visualize BERTopic documents and save as HTML.

    Args:
        model: BERTopic model
        documents: list of documents/abstracts
        reduced_embeddings: reduced embeddings used for visualization
        save_path: path to save HTML file
    """
    fig = model.visualize_documents(
        documents,
        reduced_embeddings=reduced_embeddings,
        width=1200,
        hide_annotations=True
    )
    fig.update_layout(font=dict(size=8))

    if save_path:
        fig.write_html(save_path)
        print(f"✅ Document visualization saved as HTML: {save_path}")

    plt.close()



def get_cluster_info(model, documents):
    """Return a dict with each cluster's top sentences and topic labels"""
    clusters = {}
    topics = model.get_document_info(documents)
    for topic_id in topics['Topic'].unique():
        if topic_id == -1:
            continue
        cluster_docs = topics[topics.Topic==topic_id]['Document'].tolist()
        label = model.get_topic(topic_id)
        clusters[topic_id] = {
            "top_sentences": cluster_docs[:5],  # top 5 representative sentences
            "topic_label": model.get_topic_info().loc[model.get_topic_info().Topic==topic_id, "Name"].values[0]
        }
    return clusters



# ==================== Config ====================
embedding_models = {
    "gte-small": "thenlper/gte-small",
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    # "umap": UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    "pca": PCA(n_components=5, random_state=42)
}

clustering_models = {
    "hdbscan": HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="eom"),
    "dbscan": DBSCAN(eps=0.01, min_samples=5, metric="euclidean"),
    "kmeans": KMeans(n_clusters=10, random_state=42)
}

/usr/local/lib/python3.12/dist-packages/hdbscan/plots.py:448: SyntaxWarning: invalid escape sequence '\l'
  axis.set_ylabel('$\lambda$ value')
/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [4]:
def save_document_visualization_png(model, documents, reduced_embeddings, save_path=None):
    """
    Visualize BERTopic documents and save as PNG with arrow annotations.

    Args:
        model: BERTopic model
        documents: list of documents/abstracts
        reduced_embeddings: reduced embeddings used for visualization
        save_path: path to save PNG file
    """
    import numpy as np
    import matplotlib.pyplot as plt

    # Save as PNG using matplotlib
    if save_path:
        # Ensure the save path has .png extension
        if not save_path.lower().endswith('.png'):
            save_path = save_path.rsplit('.', 1)[0] + '.png'

        # Create matplotlib version
        plt.figure(figsize=(12, 8))

        # Get topic assignments and create color map
        topic_assignments = np.array(model.topics_)
        unique_topics = np.unique(topic_assignments)
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique_topics)))
        color_map = {topic: colors[i] for i, topic in enumerate(unique_topics)}

        # Plot points
        for topic in unique_topics:
            mask = topic_assignments == topic
            if topic == -1:  # Outliers
                plt.scatter(reduced_embeddings[mask, 0], reduced_embeddings[mask, 1],
                          c='gray', alpha=0.6, s=20, label='Outliers')
            else:
                plt.scatter(reduced_embeddings[mask, 0], reduced_embeddings[mask, 1],
                          c=[color_map[topic]], alpha=0.7, s=30, label=f'Topic {topic}')

        # Randomly select 20% of topics for labeling
        import random
        non_outlier_topics = [t for t in unique_topics if t != -1]
        num_topics_to_label = max(1, int(len(non_outlier_topics) * 0.2))
        topics_to_label = random.sample(non_outlier_topics, num_topics_to_label)

        # Add arrows pointing to cluster centers (only for selected topics)
        for topic in unique_topics:
            if topic != -1 and topic in topics_to_label:  # Only label selected topics
                mask = topic_assignments == topic
                if np.any(mask):
                    center_x = np.mean(reduced_embeddings[mask, 0])
                    center_y = np.mean(reduced_embeddings[mask, 1])

                    # Get topic words and wrap text
                    topic_words = model.get_topic(topic)
                    topic_label = ", ".join([word for word, _ in topic_words[:3]])

                    # Wrap text to limit width (approximately 15 characters per line)
                    import textwrap
                    wrapped_label = "\n".join(textwrap.wrap(topic_label, width=15))
                    final_label = f"Topic {topic}\n{wrapped_label}"

                    # Calculate direction from overall center to cluster center
                    overall_center_x = np.mean(reduced_embeddings[:, 0])
                    overall_center_y = np.mean(reduced_embeddings[:, 1])

                    direction_x = center_x - overall_center_x
                    direction_y = center_y - overall_center_y

                    # Normalize and extend (back to original distance)
                    length = np.sqrt(direction_x**2 + direction_y**2)
                    if length > 0:
                        direction_x /= length
                        direction_y /= length

                    # Place text at original distance (2.5)
                     # Set text_distance based on path
                    if "pca" in save_path.lower():
                        text_distance = 0.2
                    else:
                        text_distance = 2.5

                    text_x = center_x + direction_x * text_distance
                    text_y = center_y + direction_y * text_distance

                    # Add arrow and text
                    plt.annotate(final_label,
                               xy=(center_x, center_y),
                               xytext=(text_x, text_y),
                               arrowprops=dict(arrowstyle='->', color='black', lw=2),
                               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
                               fontsize=8, ha='center', va='center')

        plt.title('Document Visualization with Topic Clusters', fontsize=14)
        plt.xlabel('Dimension 1', fontsize=12)
        plt.ylabel('Dimension 2', fontsize=12)
        plt.grid(True, alpha=0.3)

        # Save matplotlib version
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        # plt.close()
        print(f"Document visualization saved as PNG: {save_path}")

    return None

In [5]:
import matplotlib.pyplot as plt

def save_topic_differences_png(topic_model, original_topics, label, model_dir):
    """Save topic differences as PNG instead of just displaying."""
    diff_df = topic_differences(topic_model, original_topics)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    ax.table(
        cellText=diff_df.values,
        colLabels=diff_df.columns,
        cellLoc='center',
        loc='center'
    )
    plt.tight_layout()
    save_path = os.path.join(model_dir, f"{label}_topic_differences.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"📊 Saved {label} topic differences → {save_path}")


In [6]:
results = {}

output_dir = "bertopic_outputs"
os.makedirs(output_dir, exist_ok=True)

# ==================== Main Loop ====================
for emb_name, emb_model_name in embedding_models.items():
    print(f"\n==== Embedding Model: {emb_name} ====")

    # Load embedding model
    embedding_model = SentenceTransformer(emb_model_name)
    embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

    for dim_name, dim_model in dim_reduction_models.items():
        reduced_embeddings = dim_model.fit_transform(embeddings)

        for clust_name, clust_model in clustering_models.items():

            # If KMeans, find optimal k
            if clust_name == "kmeans":
                best_k = find_best_k(reduced_embeddings, k_min=2, k_max=15)
                clust_model = KMeans(n_clusters=best_k, random_state=42)

            model_name = f"{emb_name}_{dim_name}_{clust_name}"
            print(f"\n--- {model_name} ---")

            # Create output folder
            model_dir = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)

            # Build BERTopic model
            topic_model = BERTopic(
                embedding_model=embedding_model,
                umap_model=dim_model if dim_name=="umap" else None,
                hdbscan_model=clust_model if clust_name=="hdbscan" else None,
                verbose=False
            )

            # Fit model
            topics, probs = topic_model.fit_transform(abstracts, embeddings)

            # Get topic info
            info = topic_model.get_topic_info()

            print('\nBERT Topics')
            display(info)

            # Exclude outlier topic (-1)
            num_clusters = len(info[info.Topic != -1])
            print(f"Number of clusters (excluding outliers): {num_clusters}")

            # Save original topic representations
            original_topics = deepcopy(topic_model.topic_representations_)

            # ===== Update representations =====
            topic_model.update_topics(abstracts, representation_model=KeyBERTInspired())

            # Show topic differences
            print('\nKeyBERTInspired')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "KeyBERTInspired", model_dir)


            topic_model.update_topics(abstracts, representation_model=MaximalMarginalRelevance(diversity=0.5))

            print('\nMMR')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "MMR", model_dir)


            max_len = 400
            truncated_docs = [doc[:max_len] for doc in abstracts]

            generator = pipeline("text2text-generation", model="google/flan-t5-small")
            rep_t5 = TextGeneration(generator, prompt="Topic: [KEYWORDS]\nDocs: [DOCUMENTS]")
            topic_model.update_topics(truncated_docs, representation_model=rep_t5)

            print('\nT5')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "T5", model_dir)



            prompt = """
            I have a topic that contains the following comments from reditt thread:
            [COMMENTS]

            The topic is described by the following keywords: [KEYWORDS]

            Based on the information above, extract a short topic label in the following format:
            topic: <short topic label>
            """

            # Update our topic representations using GPT-3.5
            client = openai.OpenAI(api_key=userdata.get('openaikey'))
            representation_model = OpenAI(
                client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
            )
            topic_model.update_topics(abstracts, representation_model=representation_model)

            print('\nOpen AI')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "Open AI", model_dir)


            # ===== Save visualizations =====
            save_document_visualization_png(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.png"))

            save_document_visualization(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.html"))

            topic_model.update_topics(abstracts, top_n_words=500)

            # Wordclouds for first 5 topics
            for topic_id in range(5):
                create_wordcloud(topic_model, topic_id, save_path=os.path.join(model_dir, f"wordcloud_topic{topic_id}.png"))

            # ===== Cluster info and topic labels =====
            clusters_info = get_cluster_info(topic_model, abstracts)
            # Save cluster info as JSON
            # import json
            # with open(os.path.join(model_dir, "clusters_info.json"), "w", encoding="utf-8") as f:
            #     json.dump(clusters_info, f, indent=2, ensure_ascii=False)

            # Print top topic labels
            # for cid, cinfo in clusters_info.items():
            #     print(f"Cluster {cid}: Label = {cinfo['topic_label']}")
            #     for sent in cinfo['top_sentences']:
            #         print(f"  - {sent}")

            # Path to save the cluster info
            txt_file_path = os.path.join(model_dir, "clusters_info.txt")

            with open(txt_file_path, "w", encoding="utf-8") as f:
                for cid, cinfo in clusters_info.items():
                    f.write(f"Cluster {cid}: Label = {cinfo['topic_label']}\n")
                    for sent in cinfo['top_sentences']:
                        f.write(f"  - {sent}\n")
                    f.write("\n")  # Add a newline between clusters

            print(f"✅ Cluster info saved to {txt_file_path}")


            results[model_name] = topic_model



==== Embedding Model: gte-small ====


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

📊 Saved T5 topic differences → bertopic_outputs/multilingual-e5-large_pca_kmeans/T5_topic_differences.png

Open AI


,Topic,Original,Updated
0,0,and | to | the | of | it,Advanced AI models and emotional responses in ...
1,1,context | window | 32k | 128k | for,Chat Length in Gemini Pro and User Experience
2,2,release | 4o | back | bring | please,Release of new features and improvements
3,3,you | ama | they | answer | about,Reddit AMA on Responding to Negative Comments
4,4,4o | to | it | and | writing,Creative Writing Process


📊 Saved Open AI topic differences → bertopic_outputs/multilingual-e5-large_pca_kmeans/Open AI_topic_differences.png


2025-08-22 11:27:50,257 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.


Document visualization saved as PNG: bertopic_outputs/multilingual-e5-large_pca_kmeans/document_vis.png
✅ Document visualization saved as HTML: bertopic_outputs/multilingual-e5-large_pca_kmeans/document_vis.html
✅ Cluster info saved to bertopic_outputs/multilingual-e5-large_pca_kmeans/clusters_info.txt


# Only one config

In [ ]:
# ==================== Config ====================
embedding_models = {
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    "umap": UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42),
}

clustering_models = {
    "hdbscan": HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="eom"),
}

output_dir = "bertopic_outputs"
os.makedirs(output_dir, exist_ok=True)

# ==================== Main Loop ====================
for emb_name, emb_model_name in embedding_models.items():
    print(f"\n==== Embedding Model: {emb_name} ====")

    # Load embedding model
    embedding_model = SentenceTransformer(emb_model_name)
    embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

    for dim_name, dim_model in dim_reduction_models.items():
        reduced_embeddings = dim_model.fit_transform(embeddings)

        for clust_name, clust_model in clustering_models.items():

            # If KMeans, find optimal k
            if clust_name == "kmeans":
                best_k = find_best_k(reduced_embeddings, k_min=2, k_max=15)
                clust_model = KMeans(n_clusters=best_k, random_state=42)

            model_name = f"{emb_name}_{dim_name}_{clust_name}"
            print(f"\n--- {model_name} ---")

            # Create output folder
            model_dir = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)

            # Build BERTopic model
            topic_model = BERTopic(
                embedding_model=embedding_model,
                umap_model=dim_model if dim_name=="umap" else None,
                hdbscan_model=clust_model if clust_name=="hdbscan" else None,
                verbose=False
            )

            # Fit model
            topics, probs = topic_model.fit_transform(abstracts, embeddings)

            # Get topic info
            info = topic_model.get_topic_info()

            print('\nBERT Topics')
            display(info)

            # Exclude outlier topic (-1)
            num_clusters = len(info[info.Topic != -1])
            print(f"Number of clusters (excluding outliers): {num_clusters}")

            # Save original topic representations
            original_topics = deepcopy(topic_model.topic_representations_)

            # ===== Update representations =====
            topic_model.update_topics(abstracts, representation_model=KeyBERTInspired())

            # Show topic differences
            print('\nKeyBERTInspired')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "KeyBERTInspired", model_dir)


            topic_model.update_topics(abstracts, representation_model=MaximalMarginalRelevance(diversity=0.5))

            print('\nMMR')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "MMR", model_dir)


            max_len = 400
            truncated_docs = [doc[:max_len] for doc in abstracts]

            generator = pipeline("text2text-generation", model="google/flan-t5-small")
            rep_t5 = TextGeneration(generator, prompt="Topic: [KEYWORDS]\nDocs: [DOCUMENTS]")
            topic_model.update_topics(truncated_docs, representation_model=rep_t5)

            print('\nT5')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "T5", model_dir)



            prompt = """
            I have a topic that contains the following comments from reditt thread:
            [COMMENTS]

            The topic is described by the following keywords: [KEYWORDS]

            Based on the information above, extract a short topic label in the following format:
            topic: <short topic label>
            """

            # Update our topic representations using GPT-3.5
            client = openai.OpenAI(api_key=userdata.get('openaikey'))
            representation_model = OpenAI(
                client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
            )
            topic_model.update_topics(abstracts, representation_model=representation_model)

            print('\nOpen AI')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "Open AI", model_dir)


            topic_model.get_topic_info()['Representation'].values


In [4]:
from google.colab import files
import shutil

# Folder you want to download
folder_path = "all_summaries"

# Make a zip archive
shutil.make_archive(folder_path, 'zip', folder_path)

# Download the zip
files.download(folder_path+".zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os
import itertools
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import json
from google.colab import drive
drive.mount('/content/drive')


# ==================== Config ====================
embedding_models = {
    "gte-small": "thenlper/gte-small",
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    # "umap": "umap",
    "pca": "pca"
}

clustering_models = {
    "hdbscan": "hdbscan",
    "dbscan": "dbscan",
    "kmeans": "kmeans"
}

# Root folders
output_root = "/content/drive/MyDrive/bertopic_outputs"
summary_root = "all_summaries"
os.makedirs(summary_root, exist_ok=True)

def crop_vertical_whitespace(img, threshold=0.98):
    """Dynamically crop top/bottom whitespace from an image."""
    if img.ndim == 3:
        gray = img.mean(axis=2)
    else:
        gray = img
    if gray.max() > 1.5:
        gray = gray / 255.0
    mask = gray < threshold
    rows_with_content = np.where(mask.any(axis=1))[0]
    if len(rows_with_content) == 0:
        return img
    top, bottom = rows_with_content[0], rows_with_content[-1]
    return img[top:bottom+1, :, :]

def generate_summary_page(config_name):
    config_path = os.path.join(output_root, config_name)

    # Representation model images (2x2 grid now)
    rep_models = ["KeyBERTInspired", "MMR", "T5", "Open AI"]
    rep_images = [os.path.join(config_path, f"{rm}_topic_differences.png") for rm in rep_models]

    # Wordclouds (top 4 topics only now)
    wc_images = [os.path.join(config_path, f"wordcloud_topic{i}.png") for i in range(4)]

    # Cluster visualization
    doc_vis = os.path.join(config_path, "document_vis.png")

    # Number of clusters (excluding outliers)
    cluster_file = os.path.join(config_path, "clusters_info.json")
    num_clusters = None
    if os.path.exists(cluster_file):
        with open(cluster_file, "r", encoding="utf-8") as f:
            clusters_info = json.load(f)
        num_clusters = sum(1 for cid in clusters_info if cid != "-1")

    # --- Unified grid dimensions ---
    fig = plt.figure(figsize=(14, 18))

    # --- Row 0+1: Representation model plots (2x2) ---
    for i, img_path in enumerate(rep_images):
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            img_cropped = crop_vertical_whitespace(img)

            row, col = divmod(i, 2)
            ax = plt.subplot2grid((6, 4), (row, col*2), colspan=2, rowspan=1)
            ax.imshow(img_cropped)
            ax.set_title(rep_models[i], fontsize=10)
            ax.axis("off")

    # --- Row 2: Wordclouds (1x4) ---
    for i, img_path in enumerate(wc_images):
        if os.path.exists(img_path):
            ax = plt.subplot2grid((6, 4), (2, i), colspan=1, rowspan=1)
            ax.imshow(mpimg.imread(img_path))
            ax.set_title(f"Topic {i}", fontsize=10)
            ax.axis("off")

    # --- Row 3-5: Cluster visualization (full width, 3 rows) ---
    ax = plt.subplot2grid((6, 4), (3, 0), colspan=4, rowspan=3)
    if os.path.exists(doc_vis):
        ax.imshow(mpimg.imread(doc_vis))
        title_text = "Cluster Visualization"
        if num_clusters is not None:
            title_text += f" | #Clusters (excluding outliers) = {num_clusters}"
        ax.set_title(title_text, fontsize=12)
    ax.axis("off")

    # Remove all white space between plots
    plt.subplots_adjust(wspace=0, hspace=0, top=0.92, bottom=0.02, left=0.02, right=0.98)
    plt.suptitle(f"Configuration: {config_name}", fontsize=16, y=0.97)

    # Save
    save_path = os.path.join(summary_root, f"{config_name}_summary.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"Saved summary: {save_path}")

# --- Loop over configs ---
for emb, red, clus in itertools.product(
    embedding_models.keys(),
    dim_reduction_models.keys(),
    clustering_models.keys()
):
    config_name = f"{emb}_{red}_{clus}"
    generate_summary_page(config_name)

print(f"\nAll summary images are stored in folder: {summary_root}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved summary: all_summaries/gte-small_pca_hdbscan_summary.png
Saved summary: all_summaries/gte-small_pca_dbscan_summary.png
Saved summary: all_summaries/gte-small_pca_kmeans_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_hdbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_dbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_kmeans_summary.png

All summary images are stored in folder: all_summaries


In [3]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json

# ==================== Config ====================
embedding_models = ["gte-small", "multilingual-e5-large"]
dim_reduction_models = ["umap", "pca"]
clustering_models = ["hdbscan", "dbscan", "kmeans"]

# Root folders
# Root folders
output_root = "/content/drive/MyDrive/bertopic_outputs"
summary_root = "all_summaries"
os.makedirs(summary_root, exist_ok=True)

def get_num_clusters(config_path):
    cluster_file = os.path.join(config_path, "clusters_info.json")
    if os.path.exists(cluster_file):
        with open(cluster_file, "r", encoding="utf-8") as f:
            clusters_info = json.load(f)
        return sum(1 for cid in clusters_info if cid != "-1")
    return None

def generate_combined_cluster_plot(embedding_name):
    # rows: clustering, cols: dim reduction
    fig, axes = plt.subplots(len(clustering_models), len(dim_reduction_models),
                             figsize=(12, 12))  # 3x2 grid

    for i, clus in enumerate(clustering_models):
        for j, dim_red in enumerate(dim_reduction_models):
            config_name = f"{embedding_name}_{dim_red}_{clus}"
            config_path = os.path.join(output_root, config_name)
            doc_vis_path = os.path.join(config_path, "document_vis.png")

            ax = axes[i, j] if len(clustering_models) > 1 else axes[j]

            if os.path.exists(doc_vis_path):
                img = mpimg.imread(doc_vis_path)
                ax.imshow(img)
                num_clusters = get_num_clusters(config_path)
                title_text = f"{embedding_name.upper()} + {dim_red.upper()} + {clus.upper()}"
                if num_clusters is not None:
                    title_text += f" | #Clusters={num_clusters}"
                ax.set_title(title_text, fontsize=10)
            else:
                ax.text(0.5, 0.5, f"Missing {config_name}", ha='center', va='center')

            ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(summary_root, f"{embedding_name}_all_dimred_clusters.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"Saved combined cluster plot: {save_path}")


# --- Loop over embeddings ---
for emb in embedding_models:
    generate_combined_cluster_plot(emb)

print(f"\nAll combined cluster images are stored in folder: {summary_root}")


Saved combined cluster plot: all_summaries/gte-small_all_dimred_clusters.png
Saved combined cluster plot: all_summaries/multilingual-e5-large_all_dimred_clusters.png

All combined cluster images are stored in folder: all_summaries
